# Augmentation Dataset Chim 150 loài từ file ZIP

Notebook này chạy trên **Google Colab**.

Chức năng:
- Đọc file `birds_dataset.zip` từ Google Drive.
- Chỉ augmentation thư mục `train`.
- Mỗi class trong `train` sau xử lý có đúng **300 ảnh**.
- `valid` và `test` được copy nguyên sang file zip mới.
- Không cần giải nén toàn bộ dataset ra Drive.
- Tạo file output: `birds_dataset_augmented.zip`.

Cấu trúc zip gốc nên có dạng:

```text
train/class_name/image.jpg
valid/class_name/image.jpg
test/class_name/image.jpg
```

Nếu zip bị lồng thêm thư mục như `birds_dataset/train/...`, notebook có phần tự nhận diện prefix.


## Bước 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Bước 2: Khai báo đường dẫn

Bạn chỉnh lại `ZIP_INPUT` nếu file zip của bạn nằm chỗ khác.


In [ ]:
ZIP_INPUT = "/content/drive/MyDrive/CS231/Dataset/birds_dataset.zip"
ZIP_OUTPUT = "/content/drive/MyDrive/CS231/Dataset/birds_dataset_augmented.zip"

TARGET_PER_CLASS = 300

print("ZIP_INPUT :", ZIP_INPUT)
print("ZIP_OUTPUT:", ZIP_OUTPUT)


## Bước 3: Import thư viện

In [ ]:
import os
import zipfile
import random
from pathlib import Path
from io import BytesIO
from PIL import Image, ImageEnhance
from tqdm import tqdm

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

random.seed(42)

print("Import xong!")


## Bước 4: Kiểm tra file ZIP gốc

Cell này kiểm tra zip có tồn tại và bên trong có bao nhiêu ảnh.


In [ ]:
if not os.path.exists(ZIP_INPUT):
    raise FileNotFoundError(f"Không tìm thấy file zip: {ZIP_INPUT}")

with zipfile.ZipFile(ZIP_INPUT, "r") as z:
    names = z.namelist()
    image_names = [n for n in names if n.lower().endswith(VALID_EXTENSIONS)]

print("Tổng số mục trong zip:", len(names))
print("Tổng số ảnh trong zip:", len(image_names))
print("\n20 mục đầu trong zip:")
for n in names[:20]:
    print(n)


## Bước 5: Hàm xử lý path trong ZIP

Phần này tự nhận diện xem zip có dạng:

```text
train/class/image.jpg
```

hay bị lồng:

```text
birds_dataset/train/class/image.jpg
```


In [ ]:
def normalize_path(name):
    return name.replace("\\", "/")


def is_image_file(name):
    return name.lower().endswith(VALID_EXTENSIONS)


def find_split_index(parts):
    """
    Tìm vị trí của train/valid/test trong path.
    Ví dụ:
    train/class/img.jpg -> index 0
    birds_dataset/train/class/img.jpg -> index 1
    """
    for i, part in enumerate(parts):
        if part in ["train", "valid", "test"]:
            return i
    return None


def parse_zip_path(path):
    """
    Trả về split, class_name nếu path là ảnh trong train/valid/test.
    Nếu không hợp lệ thì trả về None, None.
    """
    path = normalize_path(path)
    parts = path.split("/")
    idx = find_split_index(parts)

    if idx is None:
        return None, None

    # Cần ít nhất: split/class/file
    if len(parts) <= idx + 2:
        return None, None

    split = parts[idx]
    class_name = parts[idx + 1]
    return split, class_name


def output_path(split, class_name, file_name):
    """
    Chuẩn hóa path output để zip mới có dạng:
    train/class_name/file_name
    valid/class_name/file_name
    test/class_name/file_name
    """
    return f"{split}/{class_name}/{file_name}"


## Bước 6: Quét ZIP để lấy danh sách class train và file valid/test

In [ ]:
def scan_zip(zip_path):
    train_classes = {}
    valid_test_files = []
    ignored_files = []

    with zipfile.ZipFile(zip_path, "r") as z:
        for raw_name in z.namelist():
            name = normalize_path(raw_name)

            if name.endswith("/"):
                continue

            if not is_image_file(name):
                ignored_files.append(raw_name)
                continue

            split, class_name = parse_zip_path(name)

            if split == "train" and class_name:
                train_classes.setdefault(class_name, []).append(raw_name)
            elif split in ["valid", "test"] and class_name:
                valid_test_files.append(raw_name)
            else:
                ignored_files.append(raw_name)

    return train_classes, valid_test_files, ignored_files


train_classes, valid_test_files, ignored_files = scan_zip(ZIP_INPUT)

print("Số class train tìm thấy:", len(train_classes))
print("Số ảnh valid/test sẽ copy nguyên:", len(valid_test_files))
print("Số file bị bỏ qua:", len(ignored_files))

print("\n10 class train đầu tiên:")
for i, (cls, imgs) in enumerate(sorted(train_classes.items())[:10]):
    print(f"{i+1}. {cls}: {len(imgs)} ảnh")

if len(train_classes) == 0:
    raise ValueError("Không tìm thấy class train. Hãy kiểm tra cấu trúc zip có train/class/image.jpg không.")


## Bước 7: Kiểm tra số ảnh cần augment

Cell này chưa tạo ảnh, chỉ tính trước số ảnh cần tạo thêm.


In [ ]:
need_aug_total = 0
over_total = 0

for cls, imgs in train_classes.items():
    n = len(imgs)
    if n < TARGET_PER_CLASS:
        need_aug_total += TARGET_PER_CLASS - n
    elif n > TARGET_PER_CLASS:
        over_total += n - TARGET_PER_CLASS

print("Số class train:", len(train_classes))
print("Target mỗi class:", TARGET_PER_CLASS)
print("Tổng ảnh cần augment thêm:", need_aug_total)
print("Tổng ảnh dư sẽ lấy ngẫu nhiên bỏ bớt:", over_total)
print("Tổng ảnh train sau xử lý dự kiến:", len(train_classes) * TARGET_PER_CLASS)


## Bước 8: Hàm đọc ảnh và augmentation

In [ ]:
def read_image_from_zip(zip_file, image_name):
    try:
        with zip_file.open(image_name) as f:
            img = Image.open(BytesIO(f.read())).convert("RGB")
        return img
    except Exception as e:
        print(f"Lỗi đọc ảnh {image_name}: {e}")
        return None


def image_to_bytes(img, quality=95):
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    return buffer.read()


def random_augment(img):
    """
    Augmentation gồm:
    - Lật ngang
    - Xoay nhẹ ±8 độ
    - Thay đổi độ sáng nhẹ
    """
    w, h = img.size

    # Lật ngang
    if random.random() < 0.5:
        img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

    # Xoay nhẹ
    angle = random.uniform(-8, 8)
    img = img.rotate(
        angle,
        resample=Image.Resampling.BICUBIC,
        fillcolor=(0, 0, 0)
    )

    # Thay đổi độ sáng nhẹ
    brightness_factor = random.uniform(0.85, 1.15)
    img = ImageEnhance.Brightness(img).enhance(brightness_factor)

    return img


print("Đã tạo hàm augmentation.")


## Bước 9: Tạo file ZIP mới sau augmentation

Cell này sẽ chạy lâu nếu có nhiều ảnh cần augment. Output nằm ở `ZIP_OUTPUT`.


In [ ]:
def get_original_filename(raw_path):
    return Path(normalize_path(raw_path)).name


def create_augmented_zip(zip_input, zip_output, target_per_class=300):
    train_classes, valid_test_files, _ = scan_zip(zip_input)

    if len(train_classes) == 0:
        raise ValueError("Không tìm thấy class train trong zip input.")

    # Nếu output đã tồn tại thì xóa để tránh ghi đè lỗi
    if os.path.exists(zip_output):
        os.remove(zip_output)
        print("Đã xóa file output cũ.")

    with zipfile.ZipFile(zip_input, "r") as zin, \
         zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zout:

        # 1. Xử lý train
        for class_name, image_list in tqdm(sorted(train_classes.items()), desc="Xử lý train"):
            image_list = sorted(image_list)

            # Nếu class >= 300 ảnh: lấy ngẫu nhiên đúng 300 ảnh
            if len(image_list) >= target_per_class:
                selected_images = random.sample(image_list, target_per_class)

                for idx, img_name in enumerate(selected_images):
                    ext = Path(get_original_filename(img_name)).suffix.lower()
                    new_name = output_path("train", class_name, f"{class_name}_orig_{idx:04d}{ext}")
                    zout.writestr(new_name, zin.read(img_name))

            # Nếu class < 300 ảnh: copy ảnh gốc rồi augment thêm
            else:
                current_count = 0

                for idx, img_name in enumerate(image_list):
                    ext = Path(get_original_filename(img_name)).suffix.lower()
                    new_name = output_path("train", class_name, f"{class_name}_orig_{idx:04d}{ext}")
                    zout.writestr(new_name, zin.read(img_name))
                    current_count += 1

                aug_idx = 0

                while current_count < target_per_class:
                    src_img_name = random.choice(image_list)
                    img = read_image_from_zip(zin, src_img_name)

                    if img is None:
                        continue

                    aug_img = random_augment(img)
                    aug_data = image_to_bytes(aug_img, quality=95)

                    new_name = output_path("train", class_name, f"{class_name}_aug_{aug_idx:04d}.jpg")
                    zout.writestr(new_name, aug_data)

                    current_count += 1
                    aug_idx += 1

        # 2. Copy nguyên valid/test
        for raw_name in tqdm(valid_test_files, desc="Copy valid/test"):
            norm_name = normalize_path(raw_name)
            split, class_name = parse_zip_path(norm_name)
            file_name = Path(norm_name).name
            new_name = output_path(split, class_name, file_name)
            zout.writestr(new_name, zin.read(raw_name))

    print("Hoàn tất tạo zip mới:", zip_output)


create_augmented_zip(ZIP_INPUT, ZIP_OUTPUT, TARGET_PER_CLASS)


## Bước 10: Kiểm tra file ZIP mới

Cell này kiểm tra mỗi class trong `train` có đúng 300 ảnh không.


In [ ]:
def check_output_zip(zip_output, target_per_class=300):
    train_counts = {}
    valid_count = 0
    test_count = 0

    with zipfile.ZipFile(zip_output, "r") as z:
        names = z.namelist()

        for raw_name in names:
            name = normalize_path(raw_name)

            if name.endswith("/") or not is_image_file(name):
                continue

            split, class_name = parse_zip_path(name)

            if split == "train" and class_name:
                train_counts[class_name] = train_counts.get(class_name, 0) + 1
            elif split == "valid":
                valid_count += 1
            elif split == "test":
                test_count += 1

    print("===== KIỂM TRA ZIP OUTPUT =====")
    print("Số class train:", len(train_counts))
    print("Số ảnh valid:", valid_count)
    print("Số ảnh test :", test_count)

    all_ok = True
    wrong_classes = []

    for cls, count in sorted(train_counts.items()):
        if count != target_per_class:
            all_ok = False
            wrong_classes.append((cls, count))

    if all_ok:
        print(f"Tất cả class train đều có đúng {target_per_class} ảnh.")
    else:
        print("Có class sai số lượng:")
        for cls, count in wrong_classes[:30]:
            print(cls, count)

    print("Tổng ảnh train:", sum(train_counts.values()))


check_output_zip(ZIP_OUTPUT, TARGET_PER_CLASS)


## Bước 11: Mở thử một ảnh trong ZIP output

Dùng cell này để kiểm tra ảnh augment có đọc được không.


In [ ]:
import matplotlib.pyplot as plt

with zipfile.ZipFile(ZIP_OUTPUT, "r") as z:
    imgs = [n for n in z.namelist() if n.lower().endswith(VALID_EXTENSIONS)]
    sample_name = random.choice(imgs)
    with z.open(sample_name) as f:
        img = Image.open(BytesIO(f.read())).convert("RGB")

print("Ảnh mẫu:", sample_name)
plt.imshow(img)
plt.axis("off")
plt.show()
